# Framing Annotation — Step 2: Improve Coverage of Draft Annotation Instructions

- **Step 2.1** — Test draft annotation instructions on a small random sample and inspect results  
- **Step 2.2** — Run multiple annotation runs (briseus equivalent) to identify hard cases where the model disagrees with itself

The goal at this stage is **not** to achieve high reliability yet — it is to make sure your annotation instructions cover all cases you encounter in the data. Hard cases and mislabeled cases will tell you where to add decision rules and clarify your codebook.

## Setup

Load libraries and configure the API connection. The API key is loaded from a `.env` file — make sure you have a `.env` file in your project root with:

```
CAC_API_KEY=your_key_here
```

In [22]:
import pandas as pd
import requests
import os
import time
from dotenv import load_dotenv
from pathlib import Path 
from collections import Counter

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

print(f"API key loaded: {'YES' if api_key else 'NO — check your .env file'}")
print(f"Host: {HOST}")
print(f"Model: {MODEL}")

API key loaded: YES
Host: https://maki.uni-mannheim.de/v1
Model: ministral-3-14b


In [2]:
!echo $CAC_API_KEY

sk-T4QZ6ao-ia3njvfff13KEQ


## Step 1 Draft: Annotation Instructions

These are the **Step 1 annotation instructions** written based on the theoretical framework from Freudenthaler et al. (under review). The **Immigration and Crime Frame** (Security Threat Frame in the paper) belongs to dimension 5 of the framework — the ideological layer that links structural inequalities to attitudes and behaviors.

According to the paper, security threat frames portray racialized groups as implicitly or explicitly causing crime, justifying harsher policing and restrictive immigration policies.
elated to group 1 !! 
> **Note:** These instructions are a first draft and will be refined iteratively through Steps 2.1 and 2.2. Modify the `instructions` and `reminder` variables below after each round of inspection.

## NEW INSTRUCTION! 

In [ ]:
instructions = """Sie sind ein Experte für Inhaltsanalyse in einer Studie zur medialen Darstellung ethnischer und sozialer Gruppen in deutschen Nachrichten. Ihre Aufgabe ist es, zu klassifizieren, ob ein gegebener Nachrichtenabsatz einen Einwanderungs- und Kriminalitätsrahmen enthält und, falls ja, ob dieser negativ (Bedrohung) oder positiv (Sicherheitsgewinn/Prävention) dargestellt wird.

Sie erhalten Mehrfachabsätze aus deutschen Zeitungsartikeln. Gruppen können als [Gruppe 1] oder [andere Gruppe] bezeichnet werden. Konzentrieren Sie sich ausschließlich auf das, was explizit im Text steht. Schließen Sie keine Gruppenassoziationen, die nicht direkt ausgedrückt werden.

WICHTIG:
Ein Kriminalitäts- oder Sicherheitsrahmen liegt nur dann vor, wenn die Gruppe explizit mit Kriminalität, Sicherheit, Gefahr oder Prävention verknüpft wird.

NO_CRIME_FRAME:
Ein Absatz erhält dieses Label, wenn kein expliziter Kriminalitäts- oder Sicherheitsrahmen vorliegt. Dies gilt insbesondere wenn:
- Die Gruppe gar nicht erwähnt wird
- Kriminalität erwähnt wird, aber nicht mit der Gruppe verknüpft ist
- Die Gruppe als Opfer von Straftaten dargestellt wird
- Kriminalität auf soziale Bedingungen (Armut, Ausgrenzung etc.) zurückgeführt wird
- Der Text Kriminalitätszuschreibungen relativiert oder widerspricht („kein Generalverdacht“, „nicht alle“)
- Krieg, geopolitische Konflikte oder militärische Ereignisse beschrieben werden, ohne explizite Verknüpfung der Gruppe mit Terrorismus, Anschlägen, Gewalt gegen Zivilisten, Straftaten oder Sicherheitsbedrohungen
- Humanitäre Krisen oder Naturkatastrophen beschrieben werden
- Ein Täter erwähnt wird, aber NICHT Mitglied der Gruppe ist

CRIME_FRAME_NEG:
Ein Absatz erhält dieses Label, wenn die Gruppe als Quelle von Kriminalität oder Sicherheitsbedrohung dargestellt wird. Dies umfasst:
- Gruppe wird als Täter, Verdächtiger oder kriminell dargestellt
- Gruppe wird mit konkreten Straftaten verbunden (z.B. Diebstahl, Gewalt, Mord)
- Gruppe wird als Ursache steigender Kriminalität dargestellt
- Gruppe wird als Gefahr für öffentliche oder nationale Sicherheit dargestellt
- Migration oder Gruppenzugehörigkeit wird mit Kriminalität verknüpft
- Terrorismus oder Extremismus wird mit der Gruppe verbunden
- Physische Gewalt oder Bedrohung geht explizit von Gruppenmitgliedern aus
- Gruppe begrüßt, legitimiert, feiert, rechtfertigt oder unterstützt einen Anschlag, Terrorakt, Gewaltakt oder eine Straftat
- Gruppe stellt einen Anschlag, Terrorakt, Gewaltakt oder eine Straftat als Widerstand, heroische Operation oder legitime Handlung dar
- Gruppe erscheint als Unterstützer, Kommentator, ideologischer Bezugspunkt oder politischer Akteur, der explizit mit Zustimmung, Rechtfertigung, Feier, Unterstützung oder ideologischer Nähe zu Terrorismus, politischer Gewalt oder Straftaten verbunden wird

WICHTIG:
Die Gruppe muss klar mit der Straftat, Bedrohung, Gewaltlegitimation oder Sicherheitsgefahr verknüpft sein. Reine Erwähnung von Kriminalität reicht nicht. Eine direkte Verantwortung oder Täterschaft der Gruppe ist nicht zwingend erforderlich, wenn die Gruppe den Gewaltakt ausdrücklich legitimiert, begrüßt, feiert oder unterstützt.

CRIME_FRAME_POS:
Ein Absatz erhält dieses Label, wenn ein Sicherheits- oder Kriminalitätsrahmen vorhanden ist, aber positiv dargestellt wird. Dies umfasst:
- Maßnahmen zur Kriminalitätsbekämpfung im Zusammenhang mit der Gruppe
- Prävention (z.B. soziale Arbeit, Integration, Deradikalisierung)
- Differenzierung („kein Generalverdacht“, „nicht alle“)
- Kriminalität wird durch soziale Faktoren erklärt statt durch die Gruppe selbst
- Gruppenmitglieder wirken aktiv an Sicherheit oder Prävention mit (z.B. Polizei, Sozialarbeit)
- Maßnahmen werden als Verbesserung von Sicherheit oder Ordnung dargestellt

Entscheidungsregeln:
- Prüfen Sie zuerst, ob überhaupt ein Kriminalitäts- oder Sicherheitsrahmen vorliegt
- Wenn nein → NO_CRIME_FRAME
- Wenn ja → prüfen, ob die Darstellung negativ (Bedrohung) oder positiv (Schutz/Prävention) ist
- Wenn Gruppe als Täter, Gefahr, Unterstützer von Gewalt oder Legitimierer von Terrorismus erscheint → CRIME_FRAME_NEG
- Wenn Krieg, geopolitische Konflikte oder militärische Ereignisse vorkommen, vergeben Sie nur dann CRIME_FRAME_NEG, wenn die Gruppe explizit mit Terrorismus, Anschlägen, Gewalt gegen Zivilisten, Straftaten oder Sicherheitsbedrohungen verbunden wird
- Wenn Sicherheit, Prävention oder Differenzierung im Vordergrund steht → CRIME_FRAME_POS
- Wenn unklar oder widersprüchlich → UNCLEAR

Im Zweifelsfall:
Fragen Sie sich, ob ein Leser die Gruppe als kriminell, sicherheitsgefährdend, gewaltunterstützend oder als Sicherheitsfaktor wahrnehmen würde. Wenn nein → NO_CRIME_FRAME."""

reminder = "Vergeben Sie genau ein Label: NO_CRIME_FRAME, CRIME_FRAME_NEG oder CRIME_FRAME_POS. Achten Sie besonders darauf, dass die Gruppe explizit mit Kriminalität, Sicherheit, Terrorismus, Gewaltlegitimation oder Prävention verknüpft ist. Wenn Kriminalität erwähnt wird, aber nicht mit der Gruppe verbunden ist oder die Gruppe Opfer ist, vergeben Sie NO_CRIME_FRAME."

print("Instructions and reminder loaded.")

Instructions and reminder loaded.


Test another Instruction 

## Core Functions

Three functions power the annotation pipeline:

- `annotate()` — sends a single paragraph to the API and returns the raw model output
- `parse_label()` — extracts the label from the raw output, handling misspellings and drift
- `parse_explanation()` — extracts the explanation sentence from the raw output

**On label drift:** Small LLMs sometimes output unexpected labels (e.g. `CRIME` instead of `CRIME_FRAME`, or a misspelling). The `parse_label()` function handles this by doing a fuzzy match rather than an exact match. If more than 5% of your outputs are `UNCLEAR`, revisit the reminder and output format instructions.

In [10]:
def annotate(text, instructions, reminder, temperature=0.0):
    """Send a paragraph to the API for annotation."""
    prompt = f"{instructions}\n\nNow annotate this paragraph:\n\n{text}\n\n{reminder}\n\nRespond in this format exactly:\nLabel: <NO_CRIME_FRAME or CRIME_FRAME_NEG or CRIME_FRAME_POS>\nExplanation: <one sentence explaining your choice>"

    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": [
            {"role": "system", "content": "Sie sind ein Experte für Inhaltsanalyse. Befolgen Sie die Annotationsanweisungen genau und antworten Sie immer im angegebenen Format."},
            {"role": "user", "content": prompt}
        ]
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    for attempt in range(3):
        try:
            response = requests.post(
                f"{HOST}/chat/completions",
                json=payload,
                headers=headers,
                timeout=120
            )
            if response.status_code == 200:
                return response.json()['choices'][0]['message']['content'].strip()
            else:
                print(f"  [attempt {attempt+1}] Status {response.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] Error: {e}")
            time.sleep(5)
    return "ERROR"


def parse_label(raw_output):
    """Extract and normalize label from raw model output.
    Handles misspellings and label drift from small LLMs.
    """
    raw_upper = raw_output.upper()

    # Check NO_CRIME_FRAME first because it contains CRIME_FRAME
    if "NO_CRIME_FRAME" in raw_upper or "NO CRIME FRAME" in raw_upper:
        return "NO_CRIME_FRAME"
    elif "CRIME_FRAME_NEG" in raw_upper or "CRIME FRAME NEG" in raw_upper:
        return "CRIME_FRAME_NEG"
    elif "CRIME_FRAME_POS" in raw_upper or "CRIME FRAME POS" in raw_upper:
        return "CRIME_FRAME_POS"
    else:
        return "UNCLEAR"


def parse_explanation(raw_output):
    """Extract explanation sentence from raw model output."""
    for line in raw_output.split("\n"):
        if line.lower().startswith("explanation:"):
            return line[len("explanation:"):].strip()
    return raw_output


print("Functions loaded.")

Functions loaded.


## Step 2.1 — Test Draft Annotation Instructions

We run the annotation on a **random sample of 50 rows** from the translated dataset. This is equivalent to the `bacchuss()` call in the R vignette.

After running, visually inspect the results and note:
- Which cases were correctly labelled?
- Are there clear cases not described in your coding instructions?
- What are general problems with the instructions?

Then go back and update the `instructions` and `reminder` in the cell above.

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

SAMPLE_SIZE_2_1 = 200
RANDOM_SEED = 42

# Saving of full run result
OUTPUT_2_1 = RESULTS_DIR / f"annotation_step2_1_seed{RANDOM_SEED}_n{SAMPLE_SIZE_2_1}.csv"

# Separate continuous files for relevant cases makes it easier to define test set 
CRIME_FRAME_NEG = RESULTS_DIR / "crime_frame_neg.csv"
CRIME_FRAME_POS = RESULTS_DIR / "crime_frame_pos.csv"
UNCLEAR_FRAME = RESULTS_DIR / "unclear_frame.csv"

UNCLEAR_LABELS = ["UNCLEAR"]

# Manual testset later (n=500)
MANUAL_TESTSET = RESULTS_DIR / "manual_testset.csv"
# ─────────────────────────────────────────────────────────────────────────

df = pd.read_csv("dataset/all_multi_paragraphs_2022_2026.csv")
sample_2_1 = df.sample(n=SAMPLE_SIZE_2_1, random_state=RANDOM_SEED).copy().reset_index(drop=True)

print(f"Dataset loaded {len(df)} rows total")
print(f"Sample size {len(sample_2_1)} rows")
print("Running annotation at temperature=0.0 (deterministic)\n")

Dataset loaded 659895 rows total
Sample size 200 rows
Running annotation at temperature=0.0 (deterministic)



In [12]:
# Sample from the full dataset
sample_2_1 = df.sample(
    n=SAMPLE_SIZE_2_1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

In [14]:
results_2_1 = []

for i, row in sample_2_1.iterrows():
    raw = annotate(str(row["text_block"]), instructions, reminder, temperature=0.0)
    label = parse_label(raw)
    explanation = parse_explanation(raw)

    results_2_1.append({
        "article_id":    row["article_id"],
        "par_index":     row["par_index"],
        "group":         row["group"],
        "text_block_en": row["text_block"],
        "raw_output":    raw,
        "label":         label,
        "explanation":   explanation
    })

    if (i + 1) % 10 == 0:
        n_neg = sum(1 for r in results_2_1 if r["label"] == "CRIME_FRAME_NEG")
        n_pos = sum(1 for r in results_2_1 if r["label"] == "CRIME_FRAME_POS")
        n_unclear = sum(1 for r in results_2_1 if r["label"] == "UNCLEAR")

        print(
            f"  [{i+1}/{len(sample_2_1)}] done "
            f"NEG: {n_neg}, POS: {n_pos}, UNCLEAR: {n_unclear}"
        )

annotation_2_1 = pd.DataFrame(results_2_1)

# Save full Step 2.1 output only
annotation_2_1.to_csv(OUTPUT_2_1, index=False)

print(f"\nSaved Step 2.1 to {OUTPUT_2_1}")
print("\nDone!")
print(annotation_2_1["label"].value_counts())

crime_total = annotation_2_1["label"].isin(["CRIME_FRAME_NEG", "CRIME_FRAME_POS"]).sum()

print(f"\nAny CRIME_FRAME rate: {crime_total / len(annotation_2_1) * 100:.1f}%")
print(f"CRIME_FRAME_NEG rate: {(annotation_2_1['label'] == 'CRIME_FRAME_NEG').mean() * 100:.1f}%")
print(f"CRIME_FRAME_POS rate: {(annotation_2_1['label'] == 'CRIME_FRAME_POS').mean() * 100:.1f}%")
print(f"NO_CRIME_FRAME rate: {(annotation_2_1['label'] == 'NO_CRIME_FRAME').mean() * 100:.1f}%")
print(f"UNCLEAR rate: {(annotation_2_1['label'] == 'UNCLEAR').mean() * 100:.1f}%")

  [10/200] done NEG: 1, POS: 0, UNCLEAR: 0
  [20/200] done NEG: 1, POS: 0, UNCLEAR: 0
  [30/200] done NEG: 1, POS: 0, UNCLEAR: 0
  [40/200] done NEG: 3, POS: 0, UNCLEAR: 0
  [50/200] done NEG: 3, POS: 0, UNCLEAR: 0
  [60/200] done NEG: 4, POS: 0, UNCLEAR: 0
  [70/200] done NEG: 4, POS: 0, UNCLEAR: 0
  [80/200] done NEG: 4, POS: 0, UNCLEAR: 0
  [90/200] done NEG: 4, POS: 0, UNCLEAR: 0
  [100/200] done NEG: 4, POS: 0, UNCLEAR: 0
  [110/200] done NEG: 4, POS: 0, UNCLEAR: 0
  [120/200] done NEG: 5, POS: 0, UNCLEAR: 0
  [130/200] done NEG: 5, POS: 0, UNCLEAR: 0
  [140/200] done NEG: 5, POS: 0, UNCLEAR: 0
  [150/200] done NEG: 5, POS: 0, UNCLEAR: 0
  [160/200] done NEG: 5, POS: 0, UNCLEAR: 0
  [170/200] done NEG: 5, POS: 0, UNCLEAR: 0
  [180/200] done NEG: 5, POS: 0, UNCLEAR: 0
  [190/200] done NEG: 6, POS: 0, UNCLEAR: 0
  [200/200] done NEG: 6, POS: 0, UNCLEAR: 0

Saved Step 2.1 to results/annotation_step2_1_seed42_n200.csv

Done!
label
NO_CRIME_FRAME     194
CRIME_FRAME_NEG      6
Name: co

### Inspect Results

Check the label distribution and flag any `UNCLEAR` labels (label drift). If more than **5% of labels are UNCLEAR**, adjust the reminder and output format instructions and rerun.

Use the output below to visually inspect cases — especially CRIME_FRAME cases — and ask:
- Does the label match what you would expect?
- Are there edge cases your instructions don't cover yet?

In [15]:
# Label distribution
total = len(annotation_2_1)
print("=== Label Distribution ===")
for label, count in annotation_2_1["label"].value_counts().items():
    print(f"  {label}: {count} ({count/total*100:.1f}%)")

# Flag unclear labels
unclear = annotation_2_1[annotation_2_1["label"] == "UNCLEAR"]

if len(unclear) > 0:
    pct = len(unclear) / total * 100
    print(f"\nUNCLEAR labels: {len(unclear)} ({pct:.1f}%)")

    if pct > 5:
        print("   → More than 5% unclear. Consider adjusting the reminder and output format.")
else:
    print("\n✓ No unclear labels.")

=== Label Distribution ===
  NO_CRIME_FRAME: 194 (97.0%)
  CRIME_FRAME_NEG: 6 (3.0%)

✓ No unclear labels.


In [17]:
# Browse all results — sort by label for easier inspection
annotation_2_1[
    ["group", "label", "explanation", "text_block_en"]
].sort_values("label")

,group,label,explanation,text_block_en
3,MIGR,CRIME_FRAME_NEG,Der Absatz verknüpft explizit die **[Andere Gr...,[Andere Gruppe] entsorgen eigene Papiere\n\nDi...
57,ISLMST,CRIME_FRAME_NEG,Der Text verknüpft explizit **[Gruppe 1]** (Ha...,Doch dieser Anschlag ist gefühlt anders als di...
32,AFR,CRIME_FRAME_NEG,Der Absatz verknüpft explizit **[Andere Gruppe...,Auch Frau Frühwald ist selbstverständlich gepi...
111,MIGR,CRIME_FRAME_NEG,Die Gruppe wird explizit als Täterin dargestel...,Polizei in Nikosia (Symbolbild): Die Bereitsch...
188,REFUG,CRIME_FRAME_NEG,Der Absatz verknüpft explizit die **[Gruppe 1]...,Erst kurz vor der deutschen Grenze sei ihm auf...
...,...,...,...,...
72,GBR,NO_CRIME_FRAME,Der Absatz thematisiert ausschließlich kulture...,Am 11. August 1922 trafen sich im Obergeschoss...
73,RUS,NO_CRIME_FRAME,Der Absatz beschreibt militärische Angriffe au...,Maxim Timtschenko ist Vorstandschef des ukrain...
74,ESP,NO_CRIME_FRAME,Der Absatz behandelt ausschließlich sportliche...,Im wohl wichtigsten Qualifying des Formel-1-Ja...
76,UKR,NO_CRIME_FRAME,Der Absatz verknüpft die Gruppe **nicht expliz...,»Wir haben die ersten zwei Jahre nur Banditen ...


In [19]:
# Show only crime frame cases for focused inspection
crime_cases = annotation_2_1[
    annotation_2_1["label"].isin(["CRIME_FRAME_NEG", "CRIME_FRAME_POS"])
]

print(f"Crime frame cases: {len(crime_cases)}")

crime_cases[
    ["group", "label", "explanation", "text_block_en"]
].sort_values("label")

Crime frame cases: 6


,group,label,explanation,text_block_en
3,MIGR,CRIME_FRAME_NEG,Der Absatz verknüpft explizit die **[Andere Gr...,[Andere Gruppe] entsorgen eigene Papiere\n\nDi...
32,AFR,CRIME_FRAME_NEG,Der Absatz verknüpft explizit **[Andere Gruppe...,Auch Frau Frühwald ist selbstverständlich gepi...
37,TRKD,CRIME_FRAME_NEG,Der Absatz verknüpft **[Gruppe 1]** explizit m...,Hunderte [Gruppe 1] reisten Ende Februar in di...
57,ISLMST,CRIME_FRAME_NEG,Der Text verknüpft explizit **[Gruppe 1]** (Ha...,Doch dieser Anschlag ist gefühlt anders als di...
111,MIGR,CRIME_FRAME_NEG,Die Gruppe wird explizit als Täterin dargestel...,Polizei in Nikosia (Symbolbild): Die Bereitsch...
188,REFUG,CRIME_FRAME_NEG,Der Absatz verknüpft explizit die **[Gruppe 1]...,Erst kurz vor der deutschen Grenze sei ihm auf...


In [20]:
# Save one-shot candidates from Step 2.1

crime_neg_cases = annotation_2_1[annotation_2_1["label"] == "CRIME_FRAME_NEG"].copy()
crime_pos_cases = annotation_2_1[annotation_2_1["label"] == "CRIME_FRAME_POS"].copy()
unclear_cases = annotation_2_1[annotation_2_1["label"] == "UNCLEAR"].copy()

for data in [crime_neg_cases, crime_pos_cases, unclear_cases]:
    data["seed"] = RANDOM_SEED
    data["sample_size"] = SAMPLE_SIZE_2_1
    data["source_step"] = "2.1_one_shot"

if CRIME_FRAME_NEG.exists():
    old_neg = pd.read_csv(CRIME_FRAME_NEG)
    crime_neg_cases = pd.concat([old_neg, crime_neg_cases], ignore_index=True)

crime_neg_cases = crime_neg_cases.drop_duplicates(subset=["article_id", "par_index"])
crime_neg_cases.to_csv(CRIME_FRAME_NEG, index=False)

if CRIME_FRAME_POS.exists():
    old_pos = pd.read_csv(CRIME_FRAME_POS)
    crime_pos_cases = pd.concat([old_pos, crime_pos_cases], ignore_index=True)

crime_pos_cases = crime_pos_cases.drop_duplicates(subset=["article_id", "par_index"])
crime_pos_cases.to_csv(CRIME_FRAME_POS, index=False)

if UNCLEAR_FRAME.exists():
    old_unclear = pd.read_csv(UNCLEAR_FRAME)
    unclear_cases = pd.concat([old_unclear, unclear_cases], ignore_index=True)

unclear_cases = unclear_cases.drop_duplicates(subset=["article_id", "par_index"])
unclear_cases.to_csv(UNCLEAR_FRAME, index=False)

print(f"Saved CRIME_FRAME_NEG cases to {CRIME_FRAME_NEG}")
print(f"Saved CRIME_FRAME_POS cases to {CRIME_FRAME_POS}")
print(f"Saved UNCLEAR cases to {UNCLEAR_FRAME}")

Saved CRIME_FRAME_NEG cases to results/crime_frame_neg.csv
Saved CRIME_FRAME_POS cases to results/crime_frame_pos.csv
Saved UNCLEAR cases to results/unclear_frame.csv


## Step 2.2 — Check for Hard Cases (briseus equivalent)

This step runs the annotation **multiple times with higher temperature** (0.7) on a subset of rows. Higher temperature introduces randomness — if the model consistently agrees with itself across runs, the case is clear. If it disagrees, the case is hard and the instructions need clarification.

This is the Python equivalent of the `briseus()` function from the `bacchuss` R package.

After running, sort by agreement (lowest first) and inspect the hard cases:
- Why is the model uncertain?
- Can you add a decision rule to resolve the ambiguity?
- Are these genuinely borderline cases, or a problem with the instructions?

> **Note:** This runs `n_runs × sample_size` API calls. With n_runs=5 and sample_size=20, that is 100 calls. Adjust accordingly.

In [32]:
# ── Config ───────────────────────────────────────────────────────────────
SAMPLE_SIZE_2_2 = 20
N_RUNS = 5
TEMPERATURE = 0.7

# Seed only for selecting NO_CRIME_FRAME rows!
NO_CRIME_FILL_SEED = 99

HARD_CASES_FRAME = RESULTS_DIR / "hard_cases.csv"
OUTPUT_2_2 = RESULTS_DIR / f"annotation_step2_2_fillseed{NO_CRIME_FILL_SEED}_n{SAMPLE_SIZE_2_2}_runs{N_RUNS}.csv"
# ─────────────────────────────────────────────────────────────────────────

# Include all CRIME_FRAME_NEG, CRIME_FRAME_POS, and UNCLEAR cases from Step 2.1
interesting_cases = annotation_2_1[
    annotation_2_1["label"].isin(["CRIME_FRAME_NEG", "CRIME_FRAME_POS", "UNCLEAR"])
].copy()

# Fill remaining slots with NO_CRIME_FRAME cases
n_fill = SAMPLE_SIZE_2_2 - len(interesting_cases)

if n_fill > 0:
    no_crime_candidates = annotation_2_1[
        annotation_2_1["label"] == "NO_CRIME_FRAME"
    ].copy()

    no_crime_fill = no_crime_candidates.sample(
        n=min(n_fill, len(no_crime_candidates)),
        random_state=NO_CRIME_FILL_SEED
    ).copy()

    sample_2_2 = pd.concat([interesting_cases, no_crime_fill], ignore_index=True)
else:
    no_crime_fill = pd.DataFrame()
    sample_2_2 = interesting_cases.copy()

sample_2_2 = sample_2_2.drop_duplicates(
    subset=["article_id", "par_index"]
).reset_index(drop=True)

print(f"Interesting cases included: {len(interesting_cases)}")
print(f"NO_CRIME_FRAME fill cases included: {len(no_crime_fill)}")
print(f"Sample size: {len(sample_2_2)} rows")
print(f"Runs per paragraph: {N_RUNS}")
print(f"Total API calls: {len(sample_2_2) * N_RUNS}")
print(f"Temperature: {TEMPERATURE}")
print(sample_2_2["label"].value_counts())

Interesting cases included: 6
NO_CRIME_FRAME fill cases included: 14
Sample size: 20 rows
Runs per paragraph: 5
Total API calls: 100
Temperature: 0.7
label
NO_CRIME_FRAME     14
CRIME_FRAME_NEG     6
Name: count, dtype: int64


In [ ]:
results_2_2 = []

for i, row in sample_2_2.iterrows():
    text = str(row["text_block_en"])
    run_labels = [] 
    run_explanations = []

    for run in range(N_RUNS):
        raw = annotate(text, instructions, reminder, temperature=TEMPERATURE)
        run_labels.append(parse_label(raw))
        run_explanations.append(parse_explanation(raw))

    label_counts = Counter(run_labels)
    majority_label, majority_count = label_counts.most_common(1)[0]
    agreement = majority_count / N_RUNS

    result = {
        "article_id": row["article_id"],
        "par_index": row["par_index"],
        "text_block_en": text,
        "step2_1_label": row["label"],
        "final_label": majority_label,
        "agreement": agreement,
    }

    for r, lbl in enumerate(run_labels):
        result[f"run_{r+1}"] = lbl
        result[f"explanation_{r+1}"] = run_explanations[r]

    results_2_2.append(result)

    print(
        f"  [{i+1}/{len(sample_2_2)}] "
        f"{majority_label} "
        f"(agreement: {agreement:.0%}) | runs: {run_labels}"
    )

annotation_2_2 = pd.DataFrame(results_2_2).sort_values("agreement").reset_index(drop=True)

annotation_2_2.to_csv(OUTPUT_2_2, index=False)

print(f"\nStep 2.2 complete. Saved to {OUTPUT_2_2}")
print(annotation_2_2["final_label"].value_counts())

  [1/20] CRIME_FRAME_NEG (agreement: 100%) | runs: ['CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG']
  [2/20] CRIME_FRAME_NEG (agreement: 100%) | runs: ['CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG']
  [3/20] CRIME_FRAME_NEG (agreement: 100%) | runs: ['CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG']
  [4/20] NO_CRIME_FRAME (agreement: 60%) | runs: ['CRIME_FRAME_NEG', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'NO_CRIME_FRAME']
  [5/20] CRIME_FRAME_NEG (agreement: 100%) | runs: ['CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG']
  [6/20] CRIME_FRAME_NEG (agreement: 80%) | runs: ['CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG']
  [7/20] NO_CRIME_FRAME (agreement: 100%) | runs: ['NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'N

### Inspect Hard Cases

Cases with **agreement < 80%** are hard cases — the model is uncertain and your instructions likely need a clearer decision rule for these.

For each hard case, ask:
1. What makes this paragraph ambiguous?
2. Can you write a decision rule that resolves the ambiguity?
3. Is this a genuine borderline case that needs a new label or sub-category?

Then go back to the **Annotation Instructions** cell above, update the instructions, and rerun both 2.1 and 2.2.

In [ ]:
run_cols = [f"run_{r+1}" for r in range(N_RUNS)]
explanation_cols = [f"explanation_{r+1}" for r in range(N_RUNS)]

print("=== Full Results (sorted by agreement, lowest first) ===")

annotation_2_2[
    ["agreement", "step2_1_label", "final_label", "text_block_en"] 
    + run_cols 
    + explanation_cols
].sort_values("agreement")

=== Full Results (sorted by agreement, lowest first) ===


,agreement,step2_1_label,final_label,group,text_block_en,run_1,run_2,run_3,run_4,run_5,explanation_1,explanation_2,explanation_3,explanation_4,explanation_5
0,0.6,CRIME_FRAME_NEG,NO_CRIME_FRAME,ISLMST,Doch dieser Anschlag ist gefühlt anders als di...,CRIME_FRAME_NEG,NO_CRIME_FRAME,NO_CRIME_FRAME,CRIME_FRAME_NEG,NO_CRIME_FRAME,**Label: CRIME_FRAME_NEG**\n**Explanation:** D...,Der Absatz verknüpft zwar die [Gruppe 1] mit T...,Der Absatz verknüpft zwar die **„Andere Gruppe...,"Der Text verknüpft **Gruppe 1** (z. B. Hamas, ...",Der Text verknüpft zwar die Gruppe **[Andere G...
1,0.8,CRIME_FRAME_NEG,CRIME_FRAME_NEG,REFUG,Erst kurz vor der deutschen Grenze sei ihm auf...,CRIME_FRAME_NEG,CRIME_FRAME_NEG,NO_CRIME_FRAME,CRIME_FRAME_NEG,CRIME_FRAME_NEG,Der Absatz verbindet explizit die **[Gruppe 1]...,Der Absatz verknüpft explizit die [Gruppe 1] m...,Die Gruppe [Gruppe 1] wird ausschließlich als ...,Der Absatz verknüpft explizit die **[Gruppe 1]...,Der Absatz verknüpft explizit [Gruppe 1] mit *...
2,0.8,NO_CRIME_FRAME,NO_CRIME_FRAME,REFUG,\n\n[Gruppe 1] aus der Ukraine stehen in Münch...,CRIME_FRAME_NEG,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,Der Absatz verknüpft explizit die Gruppe mit e...,**Label: NO_CRIME_FRAME**\n**Explanation:** De...,Der Absatz verknüpft die Gruppe nicht mit Krim...,Der Absatz verknüpft die Gruppe explizit mit *...,Der Text verknüpft die Gruppe aus der Ukraine ...
17,1.0,CRIME_FRAME_NEG,CRIME_FRAME_NEG,AFR,Auch Frau Frühwald ist selbstverständlich gepi...,CRIME_FRAME_NEG,CRIME_FRAME_NEG,CRIME_FRAME_NEG,CRIME_FRAME_NEG,CRIME_FRAME_NEG,**Label: CRIME_FRAME_NEG**\n**Explanation:** D...,Der Absatz verknüpft explizit **[Andere Gruppe...,Der Absatz verknüpft explizit **[Andere Gruppe...,Der Absatz verknüpft explizit **[Andere Gruppe...,**Label: CRIME_FRAME_NEG**\n**Explanation:** D...
16,1.0,CRIME_FRAME_NEG,CRIME_FRAME_NEG,TRKD,Hunderte [Gruppe 1] reisten Ende Februar in di...,CRIME_FRAME_NEG,CRIME_FRAME_NEG,CRIME_FRAME_NEG,CRIME_FRAME_NEG,CRIME_FRAME_NEG,Die Gruppe wird explizit als Teil einer *„fünf...,Die **[Gruppe 1]** wird explizit mit *„türkisc...,Der Absatz verknüpft **[Gruppe 1]** explizit m...,Der Absatz verknüpft **[Gruppe 1]** explizit m...,**Label: CRIME_FRAME_NEG**\n**Explanation:** D...
15,1.0,CRIME_FRAME_NEG,CRIME_FRAME_NEG,MIGR,Polizei in Nikosia (Symbolbild): Die Bereitsch...,CRIME_FRAME_NEG,CRIME_FRAME_NEG,CRIME_FRAME_NEG,CRIME_FRAME_NEG,CRIME_FRAME_NEG,Die **[Gruppe 1]** wird explizit als Täterin i...,Der Absatz verknüpft die Gruppe explizit mit S...,Der Text verknüpft die **[Gruppe 1]** explizit...,Die Gruppe wird explizit als Täterin dargestel...,Die Gruppe wird explizit als Täterin dargestel...
14,1.0,NO_CRIME_FRAME,NO_CRIME_FRAME,FRA,Anders als bei den Frauen ging es für die Männ...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,Der Text thematisiert einen Sportsieg und die ...,Der Absatz behandelt einen sportlichen Wettbew...,Der Absatz thematisiert ausschließlich sportli...,Der Absatz thematisiert einen Sportwettkampf (...,Der Absatz behandelt ausschließlich sportliche...
13,1.0,NO_CRIME_FRAME,NO_CRIME_FRAME,HUN,Keine Kritik an Putin\n\nDass es um Orbán eins...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,"Es wird zwar politischer Dissens erwähnt, aber...",Der Absatz behandelt ausschließlich politische...,"Der Absatz thematisiert politische Spannungen,...",Der Absatz thematisiert politische Konflikte u...,**Label: NO_CRIME_FRAME**\n**Explanation:** De...
12,1.0,NO_CRIME_FRAME,NO_CRIME_FRAME,REFUG,Wer allerdings in der Ukraine nur über eine be...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,"Der Absatz behandelt Aufenthaltsregelungen, Di...",**Label: NO_CRIME_FRAME**\n**Explanation:** De...,"Der Absatz thematisiert Aufenthaltsregelungen,...",Der Absatz behandelt ausschließlich Aspekte vo...,**Label: NO_CRIME_FRAME**\n**Explanation:** De...
11,1.0,NO_CRIME_FRAME,NO_CRIME_FRAME,HUN,[Gruppe 1

In [ ]:
hard_cases = annotation_2_2[annotation_2_2["agreement"] < 0.8].copy()

print(f"Hard cases (agreement < 80%): {len(hard_cases)} out of {len(annotation_2_2)}")

for _, row in hard_cases.iterrows():
    runs = [row[f"run_{r+1}"] for r in range(N_RUNS)]

    print(f"\n{'='*60}")
    print(f"Agreement: {row['agreement']:.0%}")
    print(f"Step 2.1 label: {row['step2_1_label']}")
    print(f"Final label: {row['final_label']}")
    print(f"Runs: {runs}")
    print(f"Text: {str(row['text_block_en'])[:500]}...")
    print("→ TODO: Why is this hard? What decision rule would resolve it?")

Hard cases (agreement < 80%): 1 out of 20

Agreement: 60%
Step 2.1 label: CRIME_FRAME_NEG
Final label: NO_CRIME_FRAME
Runs: ['CRIME_FRAME_NEG', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'NO_CRIME_FRAME']
Group: ISLMST
Text: Doch dieser Anschlag ist gefühlt anders als die vorherigen, da er in der liberalen und hedonistischen israelischen Mittelmeermetropole stattfand. Tel Aviv ist das lebensfrohe Herz Israels, das Zentrum des kleinen Landes. Ein Anschlag mitten in dieser Stadt gibt allen [Andere Gruppe] ein besonders starkes Gefühl der Verletzlichkeit. Und er macht deutlich, wie schnell die vermeintliche Sicherheit verloren gehen kann – trotz einer starken Armee, trotz exzellenter Geheimdienste, trotz verschiedener ...
→ TODO: Why is this hard? What decision rule would resolve it?


In [ ]:
#check hardcases in detail! 

row = hard_cases.iloc[0] #checks first one needs to be changed if multiple

print("ARTICLE ID:", row["article_id"])
print("PAR INDEX:", row["par_index"])
print("GROUP:", row["group"])
print("AGREEMENT:", row["agreement"])
print("STEP 2.1 LABEL:", row["step2_1_label"])
print("FINAL LABEL:", row["final_label"])

print("\nTEXT:\n")
print(row["text_block_en"])

print("\nRUNS AND EXPLANATIONS:\n")
for r in range(N_RUNS):
    print(f"\n--- Run {r+1} ---")
    print("Label:", row[f"run_{r+1}"])
    print("Explanation:", row[f"explanation_{r+1}"])

ARTICLE ID: 6327129a3dd49b919c16d336
PAR INDEX: 3
GROUP: ISLMST
AGREEMENT: 0.6
STEP 2.1 LABEL: CRIME_FRAME_NEG
FINAL LABEL: NO_CRIME_FRAME

TEXT:

Doch dieser Anschlag ist gefühlt anders als die vorherigen, da er in der liberalen und hedonistischen israelischen Mittelmeermetropole stattfand. Tel Aviv ist das lebensfrohe Herz Israels, das Zentrum des kleinen Landes. Ein Anschlag mitten in dieser Stadt gibt allen [Andere Gruppe] ein besonders starkes Gefühl der Verletzlichkeit. Und er macht deutlich, wie schnell die vermeintliche Sicherheit verloren gehen kann – trotz einer starken Armee, trotz exzellenter Geheimdienste, trotz verschiedener Spezialkräfte im Kampf gegen den Terror.

Es ist Ramadan, der muslimische Fastenmonat. Schon vor Wochen bereiteten sich die israelischen Sicherheitsorgane für genau das vor, was nun eingetreten ist: eine Terrorwelle in Zeiten religiöser Feierlichkeiten. An vielen Orten im Westjordanland feierten Palästinenser gestern das Attentat. [Gruppe 1] wie die H

In [38]:
# Agreement summary
print("=== Agreement Distribution ===")
print(annotation_2_2["agreement"].describe().round(2))

print(f"\nFull agreement (1.00): {len(annotation_2_2[annotation_2_2['agreement'] == 1.0])} rows")
print(f"Hard cases (<0.80):    {len(annotation_2_2[annotation_2_2['agreement'] < 0.8])} rows")
print(f"Very hard (<0.60):     {len(annotation_2_2[annotation_2_2['agreement'] < 0.6])} rows")

print("\n=== Final Label Distribution ===")
print(annotation_2_2["final_label"].value_counts())

=== Agreement Distribution ===
count    20.00
mean      0.96
std       0.10
min       0.60
25%       1.00
50%       1.00
75%       1.00
max       1.00
Name: agreement, dtype: float64

Full agreement (1.00): 17 rows
Hard cases (<0.80):    1 rows
Very hard (<0.60):     0 rows

=== Final Label Distribution ===
final_label
NO_CRIME_FRAME     15
CRIME_FRAME_NEG     5
Name: count, dtype: int64


In [ ]:
# Save hard cases continuously - use later for test set/ annotation instruction 
hard_cases["source_step"] = "2.2_multiple_runs"
hard_cases["n_runs"] = N_RUNS
hard_cases["temperature"] = TEMPERATURE
hard_cases["fill_seed"] = NO_CRIME_FILL_SEED

if HARD_CASES_FRAME.exists():
    old_hard_cases = pd.read_csv(HARD_CASES_FRAME)
    hard_cases = pd.concat([old_hard_cases, hard_cases], ignore_index=True)

hard_cases = hard_cases.drop_duplicates(subset=["article_id", "par_index"])
hard_cases.to_csv(HARD_CASES_FRAME, index=False)

print(f"Saved hard cases to {HARD_CASES_FRAME}")
print(f"Total hard cases saved: {len(hard_cases)}")

Saved hard cases to results/hard_cases.csv
Total hard cases saved: 1


## Next Steps

After inspecting the results:

1. **Note problems** — write down what went wrong and why in a markdown cell or comment
2. **Update instructions** — go back to the instructions cell and add/clarify decision rules
3. **Repeat 2.1 and 2.2** — rerun both steps with the updated instructions
4. **Stop when** your instructions broadly cover the cases you see, even if reliability is not yet perfect

Once coverage is broad, move to **Step 3** — testing on a curated sample of hard and soft cases with human gold-standard labels, adding chain-of-thought prompting and few-shot examples.

---

### Notes on This Round (fill in as you iterate)

| Round | Problem Identified | Decision Rule Added |
|-------|-------------------|---------------------|
| 1  (Seed 42, n= 200)   |Model is unsure whether CRIME_FRAME_NEG requires the group to be the actual perpetrator, or whether it is enough that the group praises, legitimizes, supports, or is linked to a terror attack. -> Should label as CRIME_FRAME_NEG| Zusätzliche Entscheidungsregel zu Terrorismus, Anschlägen und Gewaltlegitimation: Wenn eine Gruppe einen Anschlag, Terrorakt, Gewaltakt oder eine Straftat ausdrücklich begrüßt, legitimiert, feiert, rechtfertigt oder als Widerstand bzw. heroische Operation darstellt, vergeben Sie CRIME_FRAME_NEG, auch wenn die Gruppe keine direkte Verantwortung übernimmt oder nicht eindeutig als Täter genannt wird. Dies gilt auch dann, wenn die Gruppe eher als Unterstützer, Kommentator, ideologischer Bezugspunkt oder politischer Akteur erscheint, solange sie explizit mit Zustimmung, Rechtfertigung, Feier, Unterstützung oder ideologischer Nähe zu Terrorismus, politischer Gewalt oder Straftaten verbunden wird. Krieg, geopolitische Konflikte oder militärische Ereignisse sind nur dann CRIME_FRAME_NEG, wenn die Gruppe explizit mit Terrorismus, Anschlägen, Gewalt gegen Zivilisten, Straftaten oder Sicherheitsbedrohungen verbunden wird. |
| 2     |                   |                     |
| 3     |                   |                     |